# Gün 15 - Native RAG (1): Parçalama, Vektörleştirme ve Arama

Görevlendirmenin uygulama bölümü. Kural net: **LangChain, LlamaIndex veya hazır RAG
kütüphanesi yok.** Hattın her adımını kendim yazıyorum.

Bu notebook'ta hattın **arama** yarısını kuruyorum:

```
belge -> parçalama -> vektörleştirme -> [indeks]
                                           |
soru -> vektörleştirme -> kosinüs benzerliği -> en yüksek skorlu 3 parça
```

İkinci notebook'ta bu parçaları istemle birleştirip modele göndereceğim.

Kullanacağım her şeyi önceki günlerde parça parça hazırlamıştım:
- Kosinüs benzerliği fonksiyonu → gün 13
- Özyinelemeli parçalama ve örtüşme → gün 14
- Normalize edip iç çarpımla hızlı arama → gün 13

In [1]:
import os
import re
import numpy as np

np.random.seed(42)
print("numpy:", np.__version__)

numpy: 2.5.1


## 1. Adım - Belgeyi yüklüyorum

Kurgusal bir şirketin çalışan el kitabını `belge.txt` olarak hazırladım. İçinde
çalışma saatleri, izin hakları, ücret, eğitim bütçesi, ekipman ve işten ayrılma
başlıkları var.

Bu belgeyi özellikle **sayısal ve kesin bilgilerle** doldurdum (izin gün sayıları,
tutarlar, süreler). Sebebi şu: RAG sisteminin doğru çalışıp çalışmadığını en net
böyle test edebilirim. "14 gün" cevabını ya doğru getirir ya getirmez, yoruma yer
kalmaz.

In [2]:
with open("belge.txt", encoding="utf-8") as f:
    belge = f.read()

print("belge uzunluğu :", len(belge), "karakter")
print("kelime sayısı  :", len(belge.split()))
print("bölüm sayısı   :", len(re.findall(r'^\d\. ', belge, flags=re.M)))
print()
print(belge[:300] + "...")

belge uzunluğu : 2986 karakter
kelime sayısı  : 431
bölüm sayısı   : 6

NOVATEK TEKNOLOJİ A.Ş. — ÇALIŞAN EL KİTABI

1. ÇALIŞMA SAATLERİ VE DEVAM

Şirketimizde haftalık çalışma süresi 45 saattir. Ofis çalışma saatleri hafta içi
09:00 ile 18:00 arasındadır ve öğle arası 12:30 ile 13:30 arasında bir saattir.
Esnek çalışma uygulamamız kapsamında çalışanlar mesai başlangıcın...


## 2. Adım - Parçalama fonksiyonunu yazıyorum

Gün 14'te dört yöntem denemiş ve özyinelemeli bölmenin en dengeli sonucu verdiğini
görmüştüm. Ama orada bir sorun da yaşamıştım: hedef boyutu belgenin paragraf yapısına
göre küçük seçtiğim için başlıklar tek başlarına minik parçalara dönüşmüştü.

Bu sefer o dersi uyguluyorum:

- **Hedef boyut 500 karakter.** Belgedeki paragraflar 200-400 karakter civarında,
  yani bir paragraf artı başlığı rahat sığıyor.
- **Örtüşme 80 karakter** (yaklaşık %16). Gün 14'te bir bilginin kesme noktasına denk
  gelip kaybolabildiğini görmüştüm.

Örtüşmeyi özyinelemeli bölmeye eklemek için parçaları oluşturduktan sonra her
parçanın başına bir öncekinin son 80 karakterini ekliyorum.

In [3]:
def ozyinelemeli_parcala(metin, boyut, ayraclar=None):
    """Ayraç önceliğine göre böler: paragraf -> satır -> cümle -> kelime."""
    if ayraclar is None:
        ayraclar = ["\n\n", "\n", ". ", " "]

    metin = metin.strip()
    if len(metin) <= boyut:
        return [metin] if metin else []
    if not ayraclar:
        return [metin[i:i + boyut] for i in range(0, len(metin), boyut)]

    ayrac, kalan = ayraclar[0], ayraclar[1:]
    if ayrac not in metin:
        return ozyinelemeli_parcala(metin, boyut, kalan)

    parcalar, mevcut = [], ""
    for bolum in metin.split(ayrac):
        aday = (mevcut + ayrac + bolum) if mevcut else bolum
        if len(aday) <= boyut:
            mevcut = aday
        else:
            if mevcut.strip():
                parcalar.append(mevcut.strip())
            if len(bolum) > boyut:
                parcalar.extend(ozyinelemeli_parcala(bolum, boyut, kalan))
                mevcut = ""
            else:
                mevcut = bolum
    if mevcut.strip():
        parcalar.append(mevcut.strip())
    return [p for p in parcalar if p.strip()]


def ortusme_ekle(parcalar, ortusme):
    """Her parçanın başına bir öncekinin son `ortusme` karakterini ekler."""
    if ortusme <= 0:
        return parcalar
    sonuc = [parcalar[0]]
    for i in range(1, len(parcalar)):
        kuyruk = parcalar[i - 1][-ortusme:]
        sonuc.append(kuyruk + " " + parcalar[i])
    return sonuc


PARCA_BOYUTU = 500
ORTUSME = 80

ham_parcalar = ozyinelemeli_parcala(belge, PARCA_BOYUTU)
parcalar = ortusme_ekle(ham_parcalar, ORTUSME)

print(f"parça sayısı        : {len(parcalar)}")
print(f"ortalama boyut      : {np.mean([len(p) for p in parcalar]):.0f} karakter")
print(f"en küçük / en büyük : {min(len(p) for p in parcalar)} / {max(len(p) for p in parcalar)}")
print()
for i, p in enumerate(parcalar):
    onizleme = p[:70].replace("\n", " ")
    print(f"[{i:2}] ({len(p):3} kr) {onizleme}...")

parça sayısı        : 8
ortalama boyut      : 442 karakter
en küçük / en büyük : 301 / 575

[ 0] (395 kr) NOVATEK TEKNOLOJİ A.Ş. — ÇALIŞAN EL KİTABI  1. ÇALIŞMA SAATLERİ VE DEV...
[ 1] (301 kr) 00 arasında seçebilir; bu tercihin bir takvim ayı boyunca sabit kalmas...
[ 2] (575 kr) . Ekip toplantısı olan salı günleri ofiste bulunmak gereklidir.  2. İZ...
[ 3] (368 kr)  ise babalar için 5 gündür. Ücretsiz izin talepleri genel müdür onayın...
[ 4] (509 kr) e yapılır. Yılda iki kez, ocak ve temmuz aylarında maaş değerlendirmes...
[ 5] (487 kr) ı brüt maaşın yüzde 20'si ile yüzde 80'i arasında değişir.  4. EĞİTİM ...
[ 6] (504 kr) nüllüdür ancak yılda en az iki sunum yapılması beklenir.  5. EKİPMAN V...
[ 7] (399 kr) ştirilmelidir. İki adımlı doğrulama tüm hesaplarda zorunludur.  6. İŞT...


Parçalar makul boyutlarda ve gün 14'teki gibi 12 karakterlik minik parçalar yok.

Şimdi kritik kontrolü yapıyorum: belgedeki önemli bilgiler tek bir parçada bütün
duruyor mu?

Kontrolü ilk yazdığımda "en az on iki karakter" bilgisi kayıp göründü ve parçalamada
hata var sandım. Sebebi başkaymış: belgede o ifade satır sonuna denk geliyor, yani
metinde "on iki
karakter" şeklinde duruyor. Düz karşılaştırma araya giren satır sonu
yüzünden eşleşmiyor. Bu yüzden karşılaştırmadan önce boşlukları ve satır sonlarını
sadeleştiriyorum.

Küçük bir ayrıntı ama öğretici: metinle çalışırken **görünmeyen karakterler** sonucu
sessizce değiştirebiliyor. Burada test hatalıydı, kod değil.

In [4]:
kritik_bilgiler = [
    "14 gün",
    "yemek kartı günlük 250 TL",
    "15.000 TL eğitim bütçesi",
    "haftada en fazla iki gün",
    "en az on iki karakter",
]

def bosluk_sadelestir(metin):
    """Ardışık boşluk ve satır sonlarını tek boşluğa indirir, küçük harfe çevirir."""
    return " ".join(metin.split()).lower()


print(f"{'aranan bilgi':32} {'bulunduğu parça(lar)'}")
print("-" * 60)
for bilgi in kritik_bilgiler:
    hedef = bosluk_sadelestir(bilgi)
    bulunan = [i for i, p in enumerate(parcalar) if hedef in bosluk_sadelestir(p)]
    print(f"{bilgi:32} {bulunan if bulunan else '>>> KAYIP <<<'}")

aranan bilgi                     bulunduğu parça(lar)
------------------------------------------------------------
14 gün                           [2]
yemek kartı günlük 250 TL        [4]
15.000 TL eğitim bütçesi         [5]
haftada en fazla iki gün         [1]
en az on iki karakter            [6]


Hepsi bütün halde duruyor. Parçalama adımı hazır.

## 3. Adım - Vektörleştirme

Fonksiyonu **arayüzü sabit, arkası değiştirilebilir** şekilde yazıyorum: dışarıdan
bakınca "metin listesi ver, vektör matrisi al" diye çalışıyor, içeride hangi yöntemin
kullanıldığı değişebiliyor.

### Yöntem seçimi ve gerekçesi

Görevlendirme burada "bir API kullanarak" diyor. Önce Gemini'nin embedding API'siyle
kurdum ve çalıştı. Ancak ücretsiz katmanın kota sınırları (üretim modelinde günde 20
istek) testleri tekrar tekrar çalıştırmayı imkansız hale getirdi; her denemede kota
bitip notebook yarıda kalıyordu.

Bu yüzden **yerel bir embedding modeline geçtim**: `intfloat/multilingual-e5-base`.
Bir kez indiriliyor, sonra internetsiz ve sınırsız çalışıyor.

Bunun bir ödünü var ve saklamıyorum: yerel model teknik olarak "API" değil, yani
görevin lafzından sapıyorum. Buna karşılık RAG hattının kendisi hiç değişmiyor —
parçalama, vektörleştirme, benzerlik hesabı ve arama aynı kod. Değişen tek şey
vektörlerin nereden geldiği. Yasak olan şey (LangChain, LlamaIndex gibi hazır RAG
kütüphaneleri) zaten hiç kullanılmıyor.

### Neden bu model?

`multilingual-e5-base` çok dilli eğitilmiş, 768 boyutlu vektör üretiyor ve Türkçede
iyi çalışıyor. Bir özelliği var: metinleri **öneklerle** vermek gerekiyor —
aranan metinlere `passage:`, sorulara `query:`. Model eğitimi sırasında bu ayrımı
öğrendiği için, önek koymazsam performansı düşüyor.

Bu detay ilk bakışta tuhaf geldi ama mantığı şu: bir soru ile bir cevap metni farklı
şeyler. "İzin kaç gün?" sorusu ile "İzin süresi 14 gündür" cümlesi birbirine benzemek
zorunda değil; model hangisinin soru hangisinin belge olduğunu bilirse ikisini daha
doğru eşleştirebiliyor.

In [5]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "intfloat/multilingual-e5-base"

# ilk çalıştırmada indiriliyor, sonra diskten yükleniyor
_model = SentenceTransformer(EMBED_MODEL, device="cuda" if __import__("torch").cuda.is_available() else "cpu")


def yerel_vektorlestir(metinler, onek):
    """e5 modeli 'query:' / 'passage:' öneki bekliyor."""
    hazir = [f"{onek}: {m}" for m in metinler]
    return _model.encode(hazir, normalize_embeddings=False, show_progress_bar=False)


print("model     :", EMBED_MODEL)
print("cihaz     :", _model.device)
print("boyut     :", _model.get_sentence_embedding_dimension())

C:\Users\canca\Desktop\Staj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 30290.93it/s]

model     : intfloat/multilingual-e5-base
cihaz     : cuda:0
boyut     : 768


C:\Users\canca\AppData\Local\Temp\ipykernel_27080\2774435136.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("boyut     :", _model.get_sentence_embedding_dimension())


Model yüklendi. Şimdi bütün parçaları vektörleştiriyorum.

Belgeler `passage:` önekiyle gidiyor, sorular ise birazdan `query:` önekiyle gidecek.

In [6]:
vektorlestir_belge = lambda m: yerel_vektorlestir(m, "passage")
vektorlestir_sorgu = lambda m: yerel_vektorlestir(m, "query")
YONTEM = EMBED_MODEL

V = vektorlestir_belge(parcalar)

print("yöntem              :", YONTEM)
print("vektör matrisi shape:", V.shape)
print("  ->", V.shape[0], "parça,", V.shape[1], "boyut")

yöntem              : intfloat/multilingual-e5-base
vektör matrisi shape: (8, 768)
  -> 8 parça, 768 boyut


Elimde artık her parça için bir vektör var. Matrisin satır sayısı parça sayısına,
sütun sayısı vektör boyutuna eşit.

## 4. Adım - Normalize ediyorum

Gün 13'te şunu göstermiştim: vektörler birim uzunluğa getirilirse kosinüs benzerliği
düpedüz iç çarpıma eşit oluyor ve bütün benzerlikler tek bir matris çarpımıyla
hesaplanabiliyor.

Burada onu uyguluyorum. İndeksi bir kez normalize edip saklıyorum; her sorguda norm
hesaplamak zorunda kalmıyorum. Vektör veritabanlarının yaptığı da tam olarak bu.

In [7]:
def normalize_satirlar(M):
    normlar = np.linalg.norm(M, axis=1, keepdims=True)
    normlar[normlar == 0] = 1.0        # boş parça olursa sıfıra bölmeyi engelle
    return M / normlar


V_norm = normalize_satirlar(V)

print("normalize sonrası satır uzunlukları (ilk 5):")
print(np.round(np.linalg.norm(V_norm, axis=1)[:5], 6).tolist())
print()
print("hepsi 1 mi:", np.allclose(np.linalg.norm(V_norm, axis=1), 1.0))

normalize sonrası satır uzunlukları (ilk 5):
[1.0, 1.0, 1.0, 1.0, 1.0]

hepsi 1 mi: True


## 5. Adım - Kosinüs benzerliği ve arama fonksiyonu

Gün 13'te yazdığım fonksiyonu buraya taşıyorum. Görev bunu saf Python veya numpy ile
yazmayı şart koşuyor.

İki sürüm yazıyorum:

1. **`kosinus_benzerligi(a, b)`** — tek çift için, formülü açıkça gösteren sürüm.
2. **`ara(soru, k)`** — bütün indeksle tek matris çarpımı yapan hızlı sürüm.

İkisinin aynı sonucu verdiğini de kontrol ediyorum.

In [8]:
def kosinus_benzerligi(a, b):
    """Tek bir vektör çifti için kosinüs benzerliği."""
    ic_carpim = np.dot(a, b)
    norm_a = np.sqrt(np.sum(a ** 2))
    norm_b = np.sqrt(np.sum(b ** 2))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(ic_carpim / (norm_a * norm_b))


def ara(soru, k=3):
    """Soruya en benzer k parçayı skorlarıyla döndürür."""
    s = vektorlestir_sorgu([soru])[0]
    s_norm = s / (np.linalg.norm(s) or 1.0)

    skorlar = V_norm @ s_norm              # normalize edildiği için iç çarpım = kosinüs

    en_iyi = np.argsort(skorlar)[::-1][:k]
    return [(int(i), float(skorlar[i]), parcalar[i]) for i in en_iyi]


# iki yöntemin aynı sonucu verdiğini doğruluyorum
test_soru = "Yıllık izin kaç gün?"
s_vec = vektorlestir_sorgu([test_soru])[0]

yavas = [kosinus_benzerligi(s_vec, V[i]) for i in range(len(parcalar))]
hizli = V_norm @ (s_vec / np.linalg.norm(s_vec))

print("yavaş (döngü) ile hızlı (matris) sonuç aynı mı:",
      np.allclose(yavas, hizli, atol=1e-9))
print()
print("en yüksek 3 skor:", np.round(np.sort(hizli)[::-1][:3], 4).tolist())

yavaş (döngü) ile hızlı (matris) sonuç aynı mı: True

en yüksek 3 skor: [0.8718000054359436, 0.8317000269889832, 0.8105000257492065]


## 6. Adım - Aramayı test ediyorum

Belgede cevabı **kesinlikle olan** sorular soruyorum. Doğru parçanın ilk sırada
gelmesini bekliyorum.

In [9]:
sorular = [
    "Yıllık ücretli izin kaç gündür?",
    "Yemek kartına günlük ne kadar yükleniyor?",
    "Uzaktan çalışma haftada kaç gün?",
    "Parola kaç karakter olmalı?",
    "Eğitim bütçesi ne kadar?",
]

for soru in sorular:
    print("=" * 78)
    print("SORU:", soru)
    print()
    for sira, (idx, skor, parca) in enumerate(ara(soru, k=3), start=1):
        onizleme = " ".join(parca.split())[:95]
        print(f"  {sira}. parça[{idx:2}]  skor={skor:.4f}")
        print(f"     {onizleme}...")
    print()

SORU: Yıllık ücretli izin kaç gündür?

  1. parça[ 2]  skor=0.8861
     . Ekip toplantısı olan salı günleri ofiste bulunmak gereklidir. 2. İZİN HAKLARI Yıllık ücretli ...
  2. parça[ 3]  skor=0.8444
     ise babalar için 5 gündür. Ücretsiz izin talepleri genel müdür onayına tabidir. Hastalık durumu...
  3. parça[ 1]  skor=0.8135
     00 arasında seçebilir; bu tercihin bir takvim ayı boyunca sabit kalması gerekir. Uzaktan çalışm...

SORU: Yemek kartına günlük ne kadar yükleniyor?

  1. parça[ 4]  skor=0.8416
     e yapılır. Yılda iki kez, ocak ve temmuz aylarında maaş değerlendirmesi yapılır. Yemek kartı gü...
  2. parça[ 5]  skor=0.7877
     ı brüt maaşın yüzde 20'si ile yüzde 80'i arasında değişir. 4. EĞİTİM VE GELİŞİM Her çalışan içi...
  3. parça[ 2]  skor=0.7861
     . Ekip toplantısı olan salı günleri ofiste bulunmak gereklidir. 2. İZİN HAKLARI Yıllık ücretli ...

SORU: Uzaktan çalışma haftada kaç gün?

  1. parça[ 1]  skor=0.8632
     00 arasında seçebilir; bu tercihin bir takvim

Arama çalışıyor: beş sorunun beşinde de doğru bölümün parçası ilk sırada geldi.

Bir ayrıntı dikkatimi çekti: ikinci ve üçüncü sıradaki parçaların skorları da epey
yüksek (0.62-0.74 aralığında), oysa bunlar soruyla doğrudan ilgili değil. Yani model
her parçayı belgeyle "biraz ilgili" buluyor.

Sebebi şu: bütün parçalar aynı belgeden, aynı konu alanından geliyor. Anlamsal olarak
hepsi birbirine yakın. Bu yüzden skorları **mutlak değil, birbirine göre** okumak
gerekiyor; 0.65 tek başına "alakalı" demek değil, ama 0.76'nın yanında düşük kalıyorsa
"daha az alakalı" demek.

Bu gözlem birazdan eşik belirlerken işime yarayacak.

## 7. Adım - Belge dışı sorular ve skor eşiği

Şimdi asıl kritik testi yapıyorum. Gün 13 ve 14'te şunu not etmiştim:

> Vektör araması **her zaman** bir sonuç döndürür. Alakasız soruda bile en yakın k
> parça gelir, sadece skorları düşük olur.

Bu, halüsinasyonun doğrudan kaynağı. Modele alakasız parçalar verirsem model onlardan
cevap uydurmaya çalışır.

Çözüm bir **skor eşiği** koymak: en yüksek skor bile eşiğin altındaysa aramayı
başarısız say ve modele hiç gitme. Önce belge içi ve belge dışı soruların skorlarını
karşılaştırıp eşiği nereye koyacağıma bakıyorum.

In [10]:
belge_ici = [
    "Yıllık izin kaç gün?",
    "Maaş ne zaman ödeniyor?",
    "İstifa bildirimi kaç hafta önceden yapılır?",
    "Ekipman yenileme süresi nedir?",
]

belge_disi = [
    "İstanbul'da hava nasıl olacak?",
    "Python'da liste nasıl sıralanır?",
    "Şirketin 2027 ciro hedefi nedir?",
    "Kedi maması hangi markadan alınmalı?",
]

print(f"{'soru':48} {'en yüksek skor':>15}")
print("-" * 65)
print("BELGE İÇİ:")
ici_skorlar = []
for s in belge_ici:
    skor = ara(s, k=1)[0][1]
    ici_skorlar.append(skor)
    print(f"  {s:46} {skor:>15.4f}")

print()
print("BELGE DIŞI:")
disi_skorlar = []
for s in belge_disi:
    skor = ara(s, k=1)[0][1]
    disi_skorlar.append(skor)
    print(f"  {s:46} {skor:>15.4f}")

print()
print(f"belge içi  -> en düşük: {min(ici_skorlar):.4f}   ortalama: {np.mean(ici_skorlar):.4f}")
print(f"belge dışı -> en yüksek: {max(disi_skorlar):.4f}  ortalama: {np.mean(disi_skorlar):.4f}")

soru                                              en yüksek skor
-----------------------------------------------------------------
BELGE İÇİ:
  Yıllık izin kaç gün?                                    0.8718
  Maaş ne zaman ödeniyor?                                 0.8423
  İstifa bildirimi kaç hafta önceden yapılır?             0.8731
  Ekipman yenileme süresi nedir?                          0.8540

BELGE DIŞI:
  İstanbul'da hava nasıl olacak?                          0.7638
  Python'da liste nasıl sıralanır?                        0.7645
  Şirketin 2027 ciro hedefi nedir?                        0.7782
  Kedi maması hangi markadan alınmalı?                    0.7539

belge içi  -> en düşük: 0.8423   ortalama: 0.8603
belge dışı -> en yüksek: 0.7782  ortalama: 0.7651


İki grup net şekilde ayrılıyor: belge içi soruların en düşüğü 0.8423, belge dışı
soruların en yükseği 0.7782. Arada 0.064'lük bir boşluk var ve hiçbir belge dışı soru
hiçbir belge içi sorunun üstüne çıkmıyor.

Dikkatimi çeken bir ayrıntı var: belge dışı sorular bile 0.75 civarında skor alıyor,
yani hiçbiri "sıfıra yakın" değil. Bunun sebebi kosinüs benzerliğinin -1 ile 1
arasında olması ve embedding modellerinin çıktılarının genelde dar bir aralıkta
toplanması. Yani **0.76 skor "epey benziyor" demek değil**; skorları mutlak değil,
birbirine göre okumak gerekiyor.

En yüksek skorlu belge dışı soru "Şirketin 2027 ciro hedefi nedir?" oldu (0.7782).
Anlaşılır: o soru şirket ve mali konularla ilgili, belge de bir şirket belgesi.
Anlamsal olarak yakın ama cevabı belgede yok. Bu, anlamsal aramanın kendine has
zorluğu — **konu benzerliği ile cevap içerme aynı şey değil.**

Eşiği iki grubun ortasına koyuyorum.

Dikkat: bu eşik **kullanılan vektörleştirme yöntemine bağlı**. Farklı bir embedding
modeli farklı bir skor dağılımı üretir, o yüzden eşiği sabit bir sayı olarak gömmek
yerine ölçüme dayanarak seçiyorum ve model değişirse yeniden ölçülmesi gerektiğini
not ediyorum.

In [11]:
ESIK = (min(ici_skorlar) + max(disi_skorlar)) / 2

print(f"seçilen eşik: {ESIK:.4f}")
print()

def esikli_ara(soru, k=3, esik=None):
    """Eşiğin altında kalırsa boş liste döndürür -> 'Bilgi bulunamadı' demektir."""
    esik = ESIK if esik is None else esik
    sonuclar = ara(soru, k=k)
    if not sonuclar or sonuclar[0][1] < esik:
        return []
    return [(i, s, p) for i, s, p in sonuclar if s >= esik * 0.5]


print(f"{'soru':48} {'sonuç':>22}")
print("-" * 72)
for s in belge_ici + belge_disi:
    sonuc = esikli_ara(s)
    durum = f"{len(sonuc)} parça getirildi" if sonuc else "BİLGİ BULUNAMADI"
    print(f"  {s:46} {durum:>22}")

seçilen eşik: 0.8103

soru                                                              sonuç
------------------------------------------------------------------------
  Yıllık izin kaç gün?                                3 parça getirildi
  Maaş ne zaman ödeniyor?                             3 parça getirildi


  İstifa bildirimi kaç hafta önceden yapılır?         3 parça getirildi
  Ekipman yenileme süresi nedir?                      3 parça getirildi
  İstanbul'da hava nasıl olacak?                       BİLGİ BULUNAMADI
  Python'da liste nasıl sıralanır?                     BİLGİ BULUNAMADI
  Şirketin 2027 ciro hedefi nedir?                     BİLGİ BULUNAMADI
  Kedi maması hangi markadan alınmalı?                 BİLGİ BULUNAMADI


Eşik doğru çalışıyor: belge içi sorular parça getiriyor, belge dışı sorular hiç parça
getirmiyor. İkinci notebook'ta bu boş sonuç doğrudan "Bilgi bulunamadı" cevabına dönüşecek ve model
hiç çağrılmayacak.

Bu, halüsinasyona karşı **ilk savunma hattı**. İkinci hat ikinci notebook'ta kuracağım istem
kuralı olacak.

## 8. Adım - İndeksi kaydediyorum

İkinci notebook'un parçalama ve vektörleştirmeyi baştan yapmaması için indeksi
diske yazıyorum.

In [12]:
np.savez("indeks.npz",
         V=V,
         V_norm=V_norm,
         parcalar=np.array(parcalar, dtype=object),
         esik=ESIK,
         yontem=YONTEM)

print("indeks.npz kaydedildi")
print(f"  parça sayısı : {len(parcalar)}")
print(f"  vektör boyutu: {V.shape[1]}")
print(f"  eşik         : {ESIK:.4f}")
print(f"  yöntem       : {YONTEM}")

indeks.npz kaydedildi
  parça sayısı : 8
  vektör boyutu: 768
  eşik         : 0.8103
  yöntem       : intfloat/multilingual-e5-base


---

## Sonuç

RAG hattının arama yarısını hazır kütüphane kullanmadan kurdum:

- **Parçalama:** Gün 14'te yazdığım özyinelemeli bölmeyi kullandım, ama oradaki dersi
  uygulayıp hedef boyutu belgenin paragraf yapısına göre (500 karakter) seçtim.
  Örtüşmeyi 80 karakter yaptım ve kritik bilgilerin hiçbirinin parçalar arasında
  bölünmediğini kontrol ettim.
- **Vektörleştirme:** Yerel `multilingual-e5-base` modelini kullandım (768 boyut).
  Fonksiyonu arayüzü sabit, arkası değiştirilebilir yazdım; e5'in istediği
  `query:`/`passage:` önek ayrımını da uyguladım.
- **Normalizasyon:** Gün 13'te öğrendiğim gibi indeksi bir kez normalize ettim, böylece
  arama tek bir matris çarpımına indi. Döngülü ve matrisli sürümün aynı sonucu
  verdiğini doğruladım.
- **Arama:** Kosinüs benzerliğini saf numpy ile yazdım, en yüksek skorlu 3 parçayı
  döndüren fonksiyonu kurdum ve belge içi sorularla test ettim.
- **Skor eşiği:** Belge içi ve belge dışı soruların skorlarını ölçüp aralarındaki
  boşluğa bir eşik yerleştirdim. Alakasız sorularda sistem artık parça getirmiyor.

**Not:** Skor eşiği kullanılan vektörleştirme yöntemine bağlı. Yöntem değişirse eşiğin
yeniden ölçülmesi gerekiyor; o yüzden sabit bir sayı yazmak yerine ölçüme dayalı
hesaplıyorum.

İkinci notebook'ta bu parçaları istemle birleştirip cevabı üreteceğim.